# IT9002 Lab 2.2: Similarity Search

**Unit 02: Text Representation**

**IT9002 Natural Language Processing**

## Objectives

By the end of this lab, the following should be clear:

- Computing cosine similarity by hand and with scikit-learn, and reading its range
- Ranking a collection of documents against a query with TF-IDF vectors and cosine similarity
- Judging search results and measuring a search with precision at 3
- Deciding a vectorizer setting from a measured change in precision at 3
- Estimating bigram probabilities from counts and generating text from them

## Data

- **BBC News dataset** (Greene and Cunningham, 2006), 2,225 articles in 5 categories, raw full-text archive, http://mlg.ucd.ie/datasets/bbc.html, provided for non-commercial and research use.

- The course file `bbc_news.csv` (4.8 MB, on Moodle).

The file `bbc_news.csv` can be downloaded from Moodle. Download it to your computer, then run the cell below. If the file is already available in the runtime, the cell loads it directly; otherwise, it opens a file-upload dialog.

In [1]:
from pathlib import Path
import pandas as pd

DATA = "bbc_news.csv"
path = next((p for p in (Path(DATA), Path("data") / DATA) if p.exists()), None)
if path is None:
    from google.colab import files
    files.upload()
    path = Path(DATA)

df = pd.read_csv(path)
print(df.shape)
df.head(3)

(2225, 3)


,category,title,text
0,business,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarner...
1,business,Dollar gains on Greenspan speech,The dollar has hit its highest level against t...
2,business,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuko...


## Setup

scikit-learn, NumPy and pandas are preinstalled in Google Colab, so the cell below does not install them there. In other environments, scikit-learn will be installed if it is not already available.

In [2]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("sklearn") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)

The cell below imports the tools used in this lab and prints the versions of pandas, NumPy and scikit-learn.

In [3]:
import re
import random
from collections import Counter

import numpy as np
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("pandas", pd.__version__, "| NumPy", np.__version__, "| scikit-learn", sklearn.__version__)

pandas 3.0.2 | NumPy 2.4.2 | scikit-learn 1.9.1


## 1. Cosine Similarity

**Cosine similarity** is the cosine of the angle between two vectors: the dot product of the vectors divided by the product of their lengths. The **dot product** adds the products of matching positions, and the **length** of a vector is the square root of the sum of its squared values.

Two vectors of 2 positions are used first, so that every step can be checked by hand: `a = (3, 1)` and `b = (1, 2)`.

The cell below computes the dot product, the two lengths and the cosine.

In [4]:
a = np.array([3, 1])
b = np.array([1, 2])
dot = a @ b
length_a = np.sqrt((a ** 2).sum())
length_b = np.sqrt((b ** 2).sum())
print(f"dot product {dot} | length of a {length_a:.3f} | length of b {length_b:.3f}")
print(f"cosine {dot / (length_a * length_b):.3f}")

dot product 5 | length of a 3.162 | length of b 2.236
cosine 0.707


The dot product is 3 × 1 + 1 × 2 = **5**, the lengths are 3.162 and 2.236, and the cosine is **0.707**, the cosine of 45°.

The cell below doubles `a` and computes the cosine again.

In [5]:
print(f"cosine of 2a and b: {(2 * a) @ b / (np.sqrt(((2 * a) ** 2).sum()) * length_b):.3f}")
print(f"Euclidean distance a-b {np.sqrt(((a - b) ** 2).sum()):.3f} | 2a-b {np.sqrt(((2 * a - b) ** 2).sum()):.3f}")

cosine of 2a and b: 0.707
Euclidean distance a-b 2.236 | 2a-b 5.000


With every count of `a` doubled, the dot product and the length of `a` are both twice as large, so the factor of 2 cancels and the cosine stays **0.707**. The Euclidean distance more than doubles, reaching 5.000 against 2.236 for `a`. Cosine similarity measures the direction of a vector and ignores its length, which is why it is used to compare documents of different lengths.

`cosine_similarity()` from scikit-learn computes the cosine between every pair of rows in a matrix. The cell below applies it to three short headlines under three weightings:

- Raw counts, with `CountVectorizer()`.

- TF-IDF weights, with `TfidfVectorizer()`.

- TF-IDF weights with the scikit-learn English stop words removed, with `TfidfVectorizer(stop_words="english")`.

In [6]:
headlines = ["the team won the cup in the final",
             "the fans sang in the rain in the park",
             "cup final won by the team"]
for label, vectorizer in (("counts", CountVectorizer()),
                          ("TF-IDF", TfidfVectorizer()),
                          ("TF-IDF, stop words removed", TfidfVectorizer(stop_words="english"))):
    print(label)
    print(cosine_similarity(vectorizer.fit_transform(headlines)).round(3))

counts
[[1.    0.713 0.764]
 [0.713 1.    0.297]
 [0.764 0.297 1.   ]]
TF-IDF
[[1.    0.569 0.715]
 [0.569 1.    0.178]
 [0.715 0.178 1.   ]]
TF-IDF, stop words removed
[[1. 0. 1.]
 [0. 1. 0.]
 [1. 0. 1.]]


The diagonal is **1** in every matrix, because every headline is identical to itself.

- **Counts:** headlines 1 and 2 score **0.713**, almost as high as 1 and 3 at 0.764, although they share only `the` and `in`, which make up about half of both headlines.

- **TF-IDF:** `the` occurs in every headline and receives the lowest weight, so 1 and 2 fall to 0.569 while 1 and 3 keep 0.715. Over three headlines the weights lie only between 1.00 and 1.69, so the common words keep much of their effect.

- **Stop words removed:** headlines 1 and 3 both reduce to `cup`, `final`, `team` and `won` and score **1**, and headline 2 shares no remaining term with either.

Count and TF-IDF vectors contain no negative values, so the cosine between two documents lies between **0**, for no term in common, and **1**, for the same proportions of every term.

## 2. A Search over the Articles

**Similarity search** treats a query as a short document: the fitted vectorizer transforms it into a vector over the same vocabulary, the cosine against every article is computed, and the articles are sorted by that score.

The query is only **transformed**, never used to **fit** the vectorizer, so its columns line up with the matrix. A query word that is not in the vocabulary is ignored.

The corpus contains some articles that are exact copies of another article. A copy would occupy two places among the top results, so the cell below removes exact duplicates before the index is built and prints how many rows remain.

In [7]:
articles = df.drop_duplicates("text").reset_index(drop=True)
print(f"{len(df) - len(articles)} exact duplicates removed, {len(articles):,} articles remain")

136 exact duplicates removed, 2,089 articles remain


**136 articles** are exact duplicates of another, and **2,089** remain.

The cell below:

- Builds the TF-IDF matrix of the articles with stop words removed, and prints its shape and the average number of distinct terms per article.

- Defines `search()`, which transforms a query with the fitted vectorizer, computes its cosine against every row, and returns the indices and scores of the top `k` articles.

- Defines `show()`, which prints the rank, score, category, title and the first words of each result.

In [8]:
tv = TfidfVectorizer(stop_words="english")
M = tv.fit_transform(articles["text"])
print("matrix:", M.shape, f"| distinct terms per article: {M.nnz / M.shape[0]:.0f}")

def search(query, vectorizer=tv, matrix=M, k=3):
    scores = cosine_similarity(vectorizer.transform([query]), matrix).ravel()
    top = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i])) for i in top]

def show(query, **params):
    print(f"Query: {query!r}")
    for rank, (i, score) in enumerate(search(query, **params), start=1):
        print(f"  {rank}. {score:.3f}  [{articles.loc[i, 'category']}] {articles.loc[i, 'title']}")
        print(f"            {articles.loc[i, 'text'][:110]}...")

show("interest rates rise")

matrix: (2089, 28980) | distinct terms per article: 143
Query: 'interest rates rise'
  1. 0.389  [business] Bank voted 8-1 for no rate change
            The decision to keep interest rates on hold at 4.75% earlier this month was passed 8-1 by the Bank of England'...
  2. 0.378  [business] US interest rates increased to 2%
            US interest rates are to rise for the fourth time in five months, in a widely anticipated move. The Federal Re...
  3. 0.317  [business] Bank set to leave rates on hold
            UK interest rates are set to remain on hold at 4.75% following the latest meeting of the Bank of England. The ...


The three highest-scoring articles are all about central-bank decisions on interest rates. The top score is about **0.39**, a strong match for a short query, because an article spreads its length over about **143 distinct terms**.

The cell below shows the tokens the vectorizer keeps from the query.

In [9]:
print(tv.build_analyzer()("interest rates rise"))

['rates', 'rise']


The scikit-learn English stop list contains `interest`, so the query vector has only **2 non-zero positions**, `rates` and `rise`. The ranking is decided by those two words.

The cell below runs a query whose words are all outside the vocabulary.

In [10]:
for word in ("iphone", "youtube", "broadband"):
    print(word, "in the vocabulary:", word in tv.vocabulary_)
print(search("iphone youtube"))

iphone in the vocabulary: False
youtube in the vocabulary: False
broadband in the vocabulary: True
[(2088, 0.0), (2087, 0.0), (2086, 0.0)]


The articles are from 2004 and 2005. `iphone` and `youtube` have no column, so every score is **0**, and the top 3 are the last 3 articles in the file, which is where sorting a list of ties places them. A search can return results only for words it has seen.

## 3. Judging the Results

Whether a result is **relevant** to a query is a judgment made by a person. **Precision at 3 (P@3)** is the share of the top 3 results that are relevant: 0, 1/3, 2/3 or 1 for one query, and its average over a set of queries measures the search as a whole.

The cell below defines 10 queries and prints the top 3 results for each one.

In [11]:
QUERIES = ["interest rates rise", "new mobile phone with camera", "oscar nominations",
           "champions league final", "election campaign tax cuts", "broadband internet speed",
           "film box office record", "olympic gold medal", "oil prices record high",
           "prime minister blair"]

for query in QUERIES:
    show(query)
    print()

Query: 'interest rates rise'
  1. 0.389  [business] Bank voted 8-1 for no rate change
            The decision to keep interest rates on hold at 4.75% earlier this month was passed 8-1 by the Bank of England'...
  2. 0.378  [business] US interest rates increased to 2%
            US interest rates are to rise for the fourth time in five months, in a widely anticipated move. The Federal Re...
  3. 0.317  [business] Bank set to leave rates on hold
            UK interest rates are set to remain on hold at 4.75% following the latest meeting of the Bank of England. The ...

Query: 'new mobile phone with camera'
  1. 0.427  [tech] 'Friends fear' with lost mobiles
            People are becoming so dependent on their mobile phones that one in three are concerned that losing their phon...
  2. 0.414  [tech] Lifestyle 'governs mobile choice'
            Faster, better or funkier hardware alone is not going to help phone firms sell more handsets, research suggest...
  3. 0.413  [tech] Lifestyle

Read the title and the first words of every result. Most queries return articles on the intended subject. The query `olympic gold medal` returns an article about the **price of gold**, because the vector records the word `gold` and not the sense in which it is used.

Judge each result as relevant (`True`) or not relevant (`False`) to its query, in rank order. Then compute precision at 3 for each query and the average over the 10 queries.

In [12]:
# TODO: Enter 3 judgments (True or False) per query in rank order, then print precision at 3 for each query and the average.


**Expected result:** The average depends on the judgments. With the tutor's judgments, 7 queries score 1, two score 2/3, and `oscar nominations` and `olympic gold medal` score 1/3, for an average of **0.80**. A different person may judge a borderline article the other way, which moves the average by about 0.03 per judgment.

## 4. Selecting a Setting for the Search

**Task:** The search should return relevant articles among its top 3 results for as many queries as possible. The setting of the vectorizer decides which terms the query and the articles can match on.

Change one decision at a time and measure its effect. The candidates are:

- `stop_words=None`, which keeps the stop words in the vocabulary.

- `ngram_range=(1, 2)`, which adds bigram columns.

- A **custom stop list**: the scikit-learn English stop words together with the words that occur in more than 30% of the articles, which appear in so many articles that they separate few of them.

For each candidate, build a new vectorizer and matrix, rerun the 10 queries, and judge only the results that changed.

The selected setting should satisfy two requirements:

- The average precision at 3 over the 10 queries should be **at least 0.8**.

- No query should lose more than **1/3** of its precision compared with the default setting.

Then provide evidence for the selected setting:

- The precision at 3 of each query under the default setting and under the selected setting.

- The number of columns in each matrix.

- The completed decision table.

The cell below:

- Builds the custom stop list and prints the words it adds.

- Defines `compare()`, which prints the number of columns, non-zero cells and the memory of each matrix, then, for each query, the top 3 titles under the candidate vectorizer where they differ from the default setting.

In [13]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

presence = CountVectorizer(stop_words="english", binary=True)
share = np.asarray(presence.fit_transform(articles["text"]).mean(axis=0)).ravel()
common = set(presence.get_feature_names_out()[share > 0.3])
custom_stop_words = list(ENGLISH_STOP_WORDS | common)
print("words added to the stop list:", sorted(common))

def megabytes(matrix):
    return (matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes) / 1e6

def compare(candidate):
    matrix = candidate.fit_transform(articles["text"])
    print(f"default:   {M.shape[1]:,} columns, {M.nnz:,} non-zero cells, {megabytes(M):.1f} MB")
    print(f"candidate: {matrix.shape[1]:,} columns, {matrix.nnz:,} non-zero cells, {megabytes(matrix):.1f} MB\n")
    for query in QUERIES:
        default = [i for i, _ in search(query)]
        new = [i for i, _ in search(query, vectorizer=candidate, matrix=matrix)]
        flag = "changed" if default != new else "same"
        print(f"{query!r} ({flag})")
        if default != new:
            for rank, i in enumerate(new, start=1):
                print(f"  {rank}. [{articles.loc[i, 'category']}] {articles.loc[i, 'title']}")
    return candidate, matrix

words added to the stop list: ['mr', 'new', 'people', 'said', 'time', 'world', 'year', 'years']


In [14]:
# TODO: Run compare() for each candidate vectorizer, judge the changed results, and print the precision at 3 of every query and the average for the selected setting.


**Expected results:**

- **Stop words kept:** the top 3 of 3 queries are reordered and the average stays at about 0.80, because a word found in most articles receives the lowest IDF.
- **Bigrams:** the top 3 of 8 queries change, and the average rises to about 0.93, at 342,635 columns.
- **Custom stop list:** the list adds `mr`, `new`, `people`, `said`, `time`, `world`, `year` and `years`. The query `new mobile phone with camera` loses `new`, and the average falls to about 0.77.

Complete the table below with **one row** for each candidate setting.

In the **Reason** column, support each decision with evidence from the precision values and the column counts.

| Setting | Columns | Average precision at 3 | Decision | Reason |
|---|---|---|---|---|
| Default: stop words removed, single terms | 28,980 | | | |
| Stop words kept | | | | |
| Terms and bigrams | | | | |
| Custom stop list | | | | |

## Exercises

### Exercise 1: Near-Duplicate Articles

Exact duplicates were removed before the index was built, but the corpus also contains articles that differ from another by a few characters. Using `cosine_similarity()` on the whole TF-IDF matrix `M`:

- Count the pairs of distinct articles with a cosine of at least 0.9.

- Print the titles of the **5 most similar pairs** with their scores.

`np.triu_indices(n, k=1)` returns the row and column indices of the cells above the diagonal, so that every pair is counted once and an article is never paired with itself.

In [15]:
# TODO: Compute the cosine between every pair of articles, count the pairs at or above 0.9, and print the 5 most similar pairs.


**Expected result:** About **28 pairs** score at least 0.9, and the most similar pairs score 1.000 with the same title on both sides: articles that differ only in spacing or in a single character, which the exact-duplicate check did not catch. A cosine threshold finds the near-duplicates that string equality misses.

### Exercise 2: Counts in Place of TF-IDF

Build the search index with `CountVectorizer(stop_words="english")` in place of `TfidfVectorizer`, rerun the 10 queries, and print the top 3 titles for each.

In a Markdown cell, name the queries whose top 3 changed and state, for one of them, which term explains the change.

In [16]:
# TODO: Build the index on raw counts, rerun the 10 queries, and print the top 3 titles for each.


**Expected result:** 7 of the 10 queries change at least one result or the order of the top 3. For `prime minister blair`, the count index returns `Prime minister's questions`, because raw counts reward the repeated `prime` and `minister` and give the rare term `blair` no extra weight, while TF-IDF ranks the articles in which `blair` is frequent.

### Exercise 3 (Extension): Character N-Grams for Misspelled Queries

A **character n-gram** is a sequence of n consecutive characters. With `analyzer="char_wb"`, `TfidfVectorizer` builds its columns from the character n-grams inside each word, padded with a space at each end, so a misspelled word still shares most of its columns with the correct one.

Build a search index with `TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)` and run 3 misspelled queries against both indexes:

- `brodband internet sped`

- `olimpic gold medel`

- `electoin campain tax cuts`

Report:

- The number of columns of the character index, compared with the 28,980 of the word index.

- The top 3 titles of each query under both indexes.

- For which queries the character index recovers relevant articles that the word index misses.

In [17]:
# TODO: Build the character n-gram index, run the 3 misspelled queries against both indexes, and print the top 3 titles of each.


**Expected results:**

- **Columns:** about 106,000, nearly 4 times the word index, because every word contributes several character n-grams.
- **Queries:** for `brodband internet sped`, the word index returns no article about broadband, because `internet` is the only query word in its vocabulary, while the character index returns two. For the other two queries both indexes return similar articles, because `gold` and `tax cuts` are spelled correctly and carry the match.

### Exercise 4 (Extension): A Bigram Language Model

A **bigram language model** estimates the probability of a word given the previous word as the count of the pair divided by the count of the previous word. Using every article in the file:

- Split each article into sentences at `.`, `?` and `!`, lowercase the text, keep alphabetic tokens (with an internal apostrophe allowed), and pad each sentence with `<s>` at the start and `</s>` at the end.

- Count the unigrams and the bigrams.

- Print the estimates of P(minister | prime), P(minister's | prime), P(rates | interest) and P(prime | the).

- Multiply the bigram estimates of `<s> the prime minister said </s>` into one sentence probability.

- Generate **5 sentences** by sampling: start at `<s>`, draw the next word with `random.choices()` weighted by the bigram counts, and stop at `</s>` or after 25 words.

In a Markdown cell, state why the sentences read well over two or three words and lose their thread over a whole sentence.

In [18]:
# TODO: Tokenize with sentence markers, count unigrams and bigrams, print the 4 estimates and the sentence probability, and generate 5 sentences.


**Expected result:** The articles give 923,956 tokens and 29,348 types. P(minister | prime) = 276 / 327 = **0.844**, P(minister's | prime) = 0.113, P(rates | interest) = 0.279 and P(prime | the) = 0.004, and the sentence probability is about **4.9 × 10⁻⁶**. The exact values depend on the sentence splitting. The generated sentences contain plausible pairs such as `prime minister`, `the government` and `he said`, but each word is chosen with knowledge of one previous word only, so a sentence changes subject every few words and rarely ends with a complete thought.